# 03 — SfM-only hyperparameter tuning and ablations

This notebook consumes a completed frozen feature cache. It must not re-extract backbone features and must not use RevisitOP labels.

In [ ]:
# Run this first in every fresh Colab runtime.  It intentionally does not use PYTHONPATH.
from datetime import datetime, timezone
from pathlib import Path
import importlib
import os
import shutil
import subprocess
import sys

# Override this only if you use a fork/private clone URL.
REPO_URL = os.environ.get('CBIR_REPO_URL', 'https://github.com/armin-faraji/LightweightCBIR.git')
REPO_REVISION = os.environ.get('CBIR_REPO_REVISION', 'main')
PROJECT_ROOT = Path('/content/lightweight-cbir')
if not PROJECT_ROOT.is_dir():
    if not REPO_URL:
        raise RuntimeError('Set CBIR_REPO_URL before first use, or clone the repository to /content/lightweight-cbir.')
    subprocess.run(['git', 'clone', REPO_URL, str(PROJECT_ROOT)], check=True)
    subprocess.run(['git', '-C', str(PROJECT_ROOT), 'checkout', REPO_REVISION], check=True)
os.chdir(PROJECT_ROOT)
required_project_files = (PROJECT_ROOT / 'pyproject.toml', PROJECT_ROOT / 'src' / 'cbir' / '__init__.py', PROJECT_ROOT / 'src' / 'cbir' / 'artifacts.py')
missing_project_files = [str(path.relative_to(PROJECT_ROOT)) for path in required_project_files if not path.is_file()]
if missing_project_files:
    raise RuntimeError('The cloned repository does not contain the Colab-ready project code: ' + ', '.join(missing_project_files) + '. Push the current repository, or set CBIR_REPO_URL/CBIR_REPO_REVISION to the matching commit.')
# Do not use an editable install here: its .pth file is not automatically
# re-read by an already-running Jupyter kernel.
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '--no-deps', '--force-reinstall', '.'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'h5py', 'scipy', 'PyYAML', 'tqdm', 'matplotlib', 'Pillow'], check=True)

importlib.invalidate_caches()
try:
    import cbir
except ModuleNotFoundError as error:
    raise RuntimeError('The package installation completed but cbir is not visible to this kernel. Restart the runtime once, then rerun this first cell.') from error
print('cbir package:', cbir.__file__)

from cbir.artifacts import create_artifact_run, make_artifact_run_id
from cbir.cloud import mount_colab_drive, runtime_report, write_runtime_report
from cbir.config import config_to_dict, load_project_config
from cbir.utils import stable_hash

PERSISTENT_ROOT = mount_colab_drive() / 'lightweight-cbir'
PERSISTENT_ROOT.mkdir(parents=True, exist_ok=True)
CONFIG_PATH = Path('configs/colab.yaml')
cfg = load_project_config(CONFIG_PATH)
environment = runtime_report(project_root=PROJECT_ROOT)
config_fingerprint = stable_hash(config_to_dict(cfg))
RUN_ID = make_artifact_run_id(notebook='03', git_sha=environment['git_sha'], config_fingerprint=config_fingerprint)
ARTIFACT_RUN = create_artifact_run(PROJECT_ROOT / 'outputs', '03', run_id=RUN_ID, metadata={'git_sha': environment['git_sha'], 'config_fingerprint': config_fingerprint})
LOCAL_OUTPUT_DIR = ARTIFACT_RUN.local_dir
DRIVE_OUTPUT_ROOT = PERSISTENT_ROOT / 'notebook_outputs'
write_runtime_report(LOCAL_OUTPUT_DIR, project_root=PROJECT_ROOT, extra={'notebook': '03', 'config': str(CONFIG_PATH)})
shutil.copy2(CONFIG_PATH, ARTIFACT_RUN.path_for('config.yaml'))
print('Project:', PROJECT_ROOT)
print('Artifacts:', LOCAL_OUTPUT_DIR)

## Restore metadata and the completed frozen feature cache

Training never reads the many cache shards directly from Drive. It stages the small SfM metadata files, restores the validated full cache to fast `/content`, and then opens the local cache reader.

In [ ]:
from cbir.cache import FeatureShardReader
from cbir.cloud import publish_file, stage_file
from cbir.data.sfm import Sfm30kMetadata
from cbir.workflow import restore_complete_sfm_cache

LOCAL_SFM_ROOT = cfg.sfm.metadata_path.parent
DRIVE_SFM_ROOT = PERSISTENT_ROOT / 'datasets' / 'sfm30k'
metadata_files = (cfg.sfm.metadata_path, cfg.sfm.names_clusters_path)
LOCAL_SFM_ROOT.mkdir(parents=True, exist_ok=True)
if not all(path is not None and path.is_file() for path in metadata_files):
    if all(path is not None and (DRIVE_SFM_ROOT / path.name).is_file() for path in metadata_files):
        for path in metadata_files:
            assert path is not None
            stage_file(DRIVE_SFM_ROOT / path.name, path)
    else:
        subprocess.run([sys.executable, 'scripts/prepare_sfm30k.py', '--config', str(CONFIG_PATH), '--image-source', 'none'], cwd=PROJECT_ROOT, check=True)
        for path in metadata_files:
            assert path is not None
            publish_file(path, DRIVE_SFM_ROOT / path.name)
if cfg.sfm.names_clusters_path is None:
    raise ValueError('configs/colab.yaml must define sfm.names_clusters_path')
metadata = Sfm30kMetadata.from_official_files(cfg.sfm.metadata_path, cfg.sfm.names_clusters_path)
cache_location = restore_complete_sfm_cache(cfg, metadata)
reader = FeatureShardReader(cache_location.local_dir)
if set(reader.image_ids) != set(metadata.image_ids()):
    raise RuntimeError('restored cache does not contain exactly the full SfM-30k protocol image IDs')
val_ids = metadata.image_ids('val')
val_cases = metadata.build_validation_cases()
ARTIFACT_RUN.write_json('cache_restore.json', {'cache_name': cache_location.cache_name, 'fingerprint': cache_location.fingerprint, 'local_dir': str(cache_location.local_dir), 'drive_dir': None if cache_location.drive_dir is None else str(cache_location.drive_dir)})
print('Using restored cache:', cache_location.local_dir)

## Establish frozen CLS baseline

This is B0. Save its SfM R@1/R@5/R@10/MRR before trained heads are compared.

In [ ]:
from cbir.evaluation import evaluate_sfm_verified_pairs, final_cls_descriptors_from_cache

baseline = final_cls_descriptors_from_cache(reader, val_ids)
baseline_report = evaluate_sfm_verified_pairs(baseline, val_ids, val_cases)
ARTIFACT_RUN.write_json('baseline_metrics.json', {
    'recall_at_1': baseline_report.recall_at_1,
    'recall_at_5': baseline_report.recall_at_5,
    'recall_at_10': baseline_report.recall_at_10,
    'mrr': baseline_report.mrr,
})
print(baseline_report)

## Causal gate ablation

Run uniform, static, and reliability modes with the same layer set/dimension/training budget. Mean-vs-guided local pooling and last-only baselines are separate explicit configs, not hidden changes.

In [ ]:
from dataclasses import replace

from cbir.cloud import publish_file
from cbir.config import config_to_dict, train_fingerprint
from cbir.fusion import ReliabilityGatedFusion
from cbir.plotting import SeriesData, plot_series
from cbir.training import HeadTrainer

# Keep all three modes fixed except for their gate rule.  Each improved
# checkpoint is copied to Drive immediately, not deferred to the final cell.
GATE_MODES_TO_RUN = ('uniform', 'static', 'reliability')
PERSISTENT_CHECKPOINT_ROOT = PERSISTENT_ROOT / 'checkpoints' / '03' / ARTIFACT_RUN.run_id

def _history_payload(history):
    return {
        'epochs': history.epochs,
        'best_epoch': history.best_epoch,
        'best_metric': history.best_metric,
        'best_checkpoint': None if history.best_checkpoint is None else str(history.best_checkpoint),
    }

def run_gate_mode(mode: str):
    if mode not in {'uniform', 'static', 'reliability'}:
        raise ValueError(f'Unsupported gate mode: {mode}')
    fusion_cfg = replace(cfg.fusion, gate_mode=mode)
    run_fingerprint = train_fingerprint(
        cache_fingerprint=reader.manifest.fingerprint,
        fusion=fusion_cfg,
        training=cfg.training,
    )
    local_run_dir = LOCAL_OUTPUT_DIR / 'runs' / mode / run_fingerprint[:12]
    persistent_checkpoint = PERSISTENT_CHECKPOINT_ROOT / mode / run_fingerprint[:12] / 'best.pt'

    def checkpoint_to_drive(local_checkpoint):
        publish_file(local_checkpoint, persistent_checkpoint)

    trainer = HeadTrainer(
        head=ReliabilityGatedFusion.from_config(fusion_cfg),
        reader=reader,
        train_pairs=metadata.train_pairs,
        fusion_config=fusion_cfg,
        training_config=cfg.training,
        validation_cases=val_cases,
        validation_image_ids=val_ids,
        output_dir=local_run_dir,
        checkpoint_callback=checkpoint_to_drive,
    )
    history = trainer.fit()
    if history.best_checkpoint is None or not persistent_checkpoint.is_file():
        raise RuntimeError(f'{mode} did not produce a durable best checkpoint')
    result = {
        'mode': mode,
        'run_fingerprint': run_fingerprint,
        'cache_fingerprint': reader.manifest.fingerprint,
        'fusion_config': config_to_dict(fusion_cfg),
        'training_config': config_to_dict(cfg.training),
        'persistent_checkpoint': str(persistent_checkpoint),
        **_history_payload(history),
    }
    ARTIFACT_RUN.write_json(f'runs/{mode}/{run_fingerprint[:12]}/run_summary.json', result)
    return history, result

histories = {}
ablation_results = {}
for gate_mode in GATE_MODES_TO_RUN:
    histories[gate_mode], ablation_results[gate_mode] = run_gate_mode(gate_mode)

if not ablation_results:
    raise RuntimeError('Set GATE_MODES_TO_RUN to at least one valid mode.')

ARTIFACT_RUN.write_json('gate_ablation.json', {
    'baseline': {
        'recall_at_1': baseline_report.recall_at_1,
        'recall_at_5': baseline_report.recall_at_5,
        'recall_at_10': baseline_report.recall_at_10,
        'mrr': baseline_report.mrr,
    },
    'runs': ablation_results,
})

validation_series = {
    mode: SeriesData(
        x=[epoch['epoch'] + 1 for epoch in history.epochs],
        y=[epoch['val_recall_at_1'] for epoch in history.epochs],
    )
    for mode, history in histories.items()
}
loss_series = {
    mode: SeriesData(
        x=[epoch['epoch'] + 1 for epoch in history.epochs],
        y=[epoch['loss'] for epoch in history.epochs],
    )
    for mode, history in histories.items()
}
figure_r1, _ = plot_series(
    validation_series,
    title='SfM validation R@1 during gate ablation',
    xlabel='Epoch',
    ylabel='R@1',
    save_path=ARTIFACT_RUN.path_for('figures/gate_ablation_validation_r1.png'),
)
figure_loss, _ = plot_series(
    loss_series,
    title='Training loss during gate ablation',
    xlabel='Epoch',
    ylabel='Symmetric InfoNCE loss',
    save_path=ARTIFACT_RUN.path_for('figures/gate_ablation_loss.png'),
)
print({mode: result['best_metric'] for mode, result in ablation_results.items()})
figure_r1

## Lock the SfM-selected checkpoint and publish outputs

Set `LOCKED_MODE` only after comparing the SfM results above. This writes a durable lock that Notebook 04 reads; it must never be changed after looking at RevisitOP results. Then run the final publish cell exactly once after all desired plots/reports have been created.

In [ ]:
from cbir.cache import sha256_file
from cbir.utils import atomic_write_json

# Change this to one completed mode, for example 'reliability', after inspecting SfM only.
LOCKED_MODE = None
if LOCKED_MODE is not None:
    if LOCKED_MODE not in ablation_results:
        raise KeyError(f'{LOCKED_MODE!r} was not run in this notebook session')
    selected = ablation_results[LOCKED_MODE]
    persistent_checkpoint = Path(selected['persistent_checkpoint'])
    if not persistent_checkpoint.is_file():
        raise FileNotFoundError(f'Durable checkpoint is missing: {persistent_checkpoint}')
    lock = {
        'selected_on': 'SfM-30k validation only',
        'notebook_run_id': ARTIFACT_RUN.run_id,
        'cache_fingerprint': reader.manifest.fingerprint,
        'mode': LOCKED_MODE,
        'run_fingerprint': selected['run_fingerprint'],
        'best_epoch': selected['best_epoch'],
        'best_metric': selected['best_metric'],
        'fusion_config': selected['fusion_config'],
        'training_config': selected['training_config'],
        'checkpoint_path': str(persistent_checkpoint),
        'checkpoint_sha256': sha256_file(persistent_checkpoint),
    }
    lock_path = PERSISTENT_ROOT / 'locked' / 'locked_run.json'
    atomic_write_json(lock_path, lock)
    ARTIFACT_RUN.write_json('locked_selection.json', lock)
    print('Locked SfM-selected checkpoint:', persistent_checkpoint)
else:
    print('No checkpoint locked yet. Set LOCKED_MODE after choosing with SfM results only.')

## Publish notebook outputs to Drive

Run this as the final cell after training and, if applicable, locking. Best checkpoints were already copied to Drive at every improvement; this publishes figures, summaries, and environment provenance.

In [ ]:
published_output = ARTIFACT_RUN.publish(DRIVE_OUTPUT_ROOT)
print('Validated notebook artifacts published to:', published_output)